<a href="https://colab.research.google.com/github/han43/training/blob/master/notebooks/basic_training_notebook_kecha.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Training a microWakeWord Model

This notebook steps you through training a basic microWakeWord model. It is intended as a **starting point** for advanced users. You should use Python 3.10.

**The model generated will most likely not be usable for everyday use; it may be difficult to trigger or falsely activates too frequently. You will most likely have to experiment with many different settings to obtain a decent model!**

In the comment at the start of certain blocks, I note some specific settings to consider modifying.

This runs on Google Colab, but is extremely slow compared to training on a local GPU. If you must use Colab, be sure to Change the runtime type to a GPU. Even then, it still slow!

At the end of this notebook, you will be able to download a tflite file. To use this in ESPHome, you need to write a model manifest JSON file. See the [ESPHome documentation](https://esphome.io/components/micro_wake_word) for the details and the [model repo](https://github.com/esphome/micro-wake-word-models/tree/main/models/v2) for examples.

In [1]:
# Installs microWakeWord. Be sure to restart the session after this is finished.
import platform

if platform.system() == "Darwin":
    # `pymicro-features` is installed from a fork to support building on macOS
    !pip install 'git+https://github.com/puddly/pymicro-features@puddly/minimum-cpp-version'

# `audio-metadata` is installed from a fork to unpin `attrs` from a version that breaks Jupyter
!pip install 'git+https://github.com/whatsnowplaying/audio-metadata@d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'

!git clone https://github.com/kahrendt/microWakeWord
!pip install -e ./microWakeWord

  Cloning https://github.com/whatsnowplaying/audio-metadata (to revision d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f) to /tmp/pip-req-build-0smputh5
  Running command git clone --filter=blob:none --quiet https://github.com/whatsnowplaying/audio-metadata /tmp/pip-req-build-0smputh5
  Running command git rev-parse -q --verify 'sha^d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f'
  Running command git fetch -q https://github.com/whatsnowplaying/audio-metadata d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Running command git checkout -q d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Resolved https://github.com/whatsnowplaying/audio-metadata to commit d4ebb238e6a401bb1a5aaaac60c9e2b3cb30929f
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
fatal: destination path 'microWakeWord' already exists and is not an empty directory.
Obtaining file:///content/microWakeWord
  Installing build dependencies ... done
  Checking if build

In [2]:
# Загружаем ваши голосовые файлы
from google.colab import files
import os, shutil

os.makedirs("generated_samples", exist_ok=True)

print("Загрузите ВСЕ 27 файлов из D:\\esp\\Kecha\\my_voice_ready\\")
uploaded = files.upload()
for fname in uploaded.keys():
    shutil.move(fname, f"generated_samples/{fname}")
print(f"Загружено: {len(uploaded)} файлов")

Загрузите ВСЕ 27 файлов из D:\esp\Kecha\my_voice_ready\


Saving 2026_08_14_09_10_30.wav to 2026_08_14_09_10_30.wav
Saving 2026_08_14_09_10_35.wav to 2026_08_14_09_10_35.wav
Saving 2026_08_14_09_10_49.wav to 2026_08_14_09_10_49.wav
Saving 2026_08_14_09_10_59.wav to 2026_08_14_09_10_59.wav
Saving 2026_08_14_09_11_06.wav to 2026_08_14_09_11_06.wav
Saving 2026_08_14_09_11_10.wav to 2026_08_14_09_11_10.wav
Saving 2026_08_14_09_11_15.wav to 2026_08_14_09_11_15.wav
Saving 2026_08_14_09_11_22.wav to 2026_08_14_09_11_22.wav
Saving 2026_08_14_09_11_27.wav to 2026_08_14_09_11_27.wav
Saving 2026_08_14_09_11_31.wav to 2026_08_14_09_11_31.wav
Saving 2026_08_14_09_11_36.wav to 2026_08_14_09_11_36.wav
Saving 2026_08_14_09_11_40.wav to 2026_08_14_09_11_40.wav
Saving 2026_08_14_09_11_44.wav to 2026_08_14_09_11_44.wav
Saving 2026_08_14_09_11_48.wav to 2026_08_14_09_11_48.wav
Saving 2026_08_14_09_11_52.wav to 2026_08_14_09_11_52.wav
Saving 2026_08_14_09_11_55.wav to 2026_08_14_09_11_55.wav
Saving 2026_08_14_09_11_59.wav to 2026_08_14_09_11_59.wav
Saving 2026_08

In [3]:
# Настройка Clips и создание датасета
import sys
sys.path.insert(0, '/content/microWakeWord')

from microwakeword.audio.clips import Clips
from microwakeword.audio.augmentation import Augmentation
from microwakeword.audio.spectrograms import SpectrogramGeneration
from mmap_ninja.ragged import RaggedMmap
import os, shutil

clips = Clips(input_directory='generated_samples', file_pattern='*.wav',
              max_clip_duration_s=None, remove_silence=False,
              random_split_seed=10, split_count=0.1)
print(f"Клипов загружено: {len(clips.clips)}")

augmenter = Augmentation(augmentation_duration_s=3.2,
    augmentation_probabilities={"SevenBandParametricEQ": 0.1, "TanhDistortion": 0.1,
                                "PitchShift": 0.1, "BandStopFilter": 0.1,
                                "AddColorNoise": 0.1, "Gain": 1.0},
    impulse_paths=[], background_paths=[])

# Создаём датасет
out_dir = 'generated_augmented_features'
if os.path.exists(out_dir): shutil.rmtree(out_dir)
os.mkdir(out_dir)
os.mkdir(os.path.join(out_dir, "training"))

sg = SpectrogramGeneration(clips=clips, augmenter=augmenter, slide_frames=10, step_ms=10)
RaggedMmap.from_generator(
    out_dir=os.path.join(out_dir, "training", 'wakeword_mmap'),
    sample_generator=sg.spectrogram_generator(split="train", repeat=10),
    batch_size=10, verbose=True)

print("Датасет создан!")

Клипов загружено: 32


0it [00:00, ?it/s]

Датасет создан!


In [4]:
# Конфиг и запуск обучения
import yaml, os, shutil

config = {"window_step_ms":10, "train_dir":"trained_models/wakeword",
    "features":[{"features_dir":"generated_augmented_features","sampling_weight":1.0,
                 "penalty_weight":1.0,"truth":True,"truncation_strategy":"truncate_start","type":"mmap"}],
    "training_steps":[3000],"positive_class_weight":[1],"negative_class_weight":[1],
    "learning_rates":[0.001],"batch_size":8,"time_mask_max_size":[0],"time_mask_count":[0],
    "freq_mask_max_size":[0],"freq_mask_count":[0],"eval_step_interval":3000,
    "clip_duration_ms":1500,"target_minimization":0.0,"minimization_metric":None,
    "maximization_metric":"recall"}
with open("training_parameters.yaml","w") as f: yaml.dump(config, f)

if os.path.exists("trained_models"): shutil.rmtree("trained_models")

!python -m microwakeword.model_train_eval --training_config='training_parameters.yaml' \
--train 1 --restore_checkpoint 0 --test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 --test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 --use_weights "best_weights" mixednet \
--pointwise_filters "64,64,64,64" --repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection "0,0,0,0" \
--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3

INFO:absl:Loading and analyzing data sets.
2026-08-14 07:27:00.577461: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786692420.578986    2005 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (8, 204, 40)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (8, 204, 1, 40

In [5]:
# Конвертируем и скачиваем модель
!python -m microwakeword.model_train_eval --training_config='training_parameters.yaml' \
--train 0 --restore_checkpoint 0 --test_tf_nonstreaming 0 --test_tflite_nonstreaming 0 \
--test_tflite_nonstreaming_quantized 0 --test_tflite_streaming 0 \
--test_tflite_streaming_quantized 1 --use_weights "last_weights" mixednet \
--pointwise_filters "64,64,64,64" --repeat_in_block "1,1,1,1" \
--mixconv_kernel_sizes '[5], [7,11], [9,15], [23]' --residual_connection "0,0,0,0" \
--first_conv_filters 32 --first_conv_kernel_size 5 --stride 3

INFO:absl:Loading and analyzing data sets.
2026-08-14 07:27:36.980599: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1786692456.982110    2229 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Model: "functional"
┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (1, 204, 40)      │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ expand_dims         │ (1, 204, 1, 40

In [6]:
from google.colab import files
files.download("trained_models/wakeword/tflite_stream_state_internal_quant/stream_state_internal_quant.tflite")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>